In [ ]:
# Jupyter Notebook: 01_eda.ipynb
# Exploratory Data Analysis for Wavelet Drift Detection Project

# Cell 1: Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from experiments.synthetic_data import SyntheticDriftGenerator

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 4)

# Cell 2: Generate Synthetic Data
print("Generating synthetic drifting streams...")
gen = SyntheticDriftGenerator(seed=42)

# Generate all drift types
drifts = {}
for drift_type in ['sudden', 'gradual', 'incremental', 'recurring']:
    X, y, true_drifts = gen.generate(drift_type=drift_type, n=5000)
    drifts[drift_type] = (X, y, true_drifts)
    print(f"✓ {drift_type}: y range [{y.min():.3f}, {y.max():.3f}]")

# Cell 3: Visualize Sudden Drift
X, y, true_drifts = drifts['sudden']

fig, axes = plt.subplots(2, 1, figsize=(14, 6))

# Time series
axes[0].plot(y, 'b-', alpha=0.7, linewidth=0.5)
for drift_t in true_drifts:
    axes[0].axvline(drift_t, color='r', linestyle='--', alpha=0.5, linewidth=2)
axes[0].set_title('Sudden Drift: Y-series with True Drift Times')
axes[0].set_ylabel('Y')
axes[0].grid(True, alpha=0.3)

# Error (first AR residual)
error = y[1:] - 0.5*y[:-1]
axes[1].plot(error, 'g-', alpha=0.7, linewidth=0.5)
for drift_t in true_drifts:
    axes[1].axvline(drift_t, color='r', linestyle='--', alpha=0.5, linewidth=2)
axes[1].set_title('Prediction Error (y_t - 0.5*y_{t-1})')
axes[1].set_ylabel('Error')
axes[1].set_xlabel('Time')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Cell 4: Compare All Drift Types
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes = axes.flatten()

for idx, (drift_type, (X, y, true_drifts)) in enumerate(drifts.items()):
    ax = axes[idx]
    ax.plot(y, 'b-', alpha=0.7, linewidth=0.5)
    for drift_t in true_drifts:
        ax.axvline(drift_t, color='r', linestyle='--', alpha=0.5)
    ax.set_title(f'{drift_type.capitalize()} Drift')
    ax.set_ylabel('Y')
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel('Time')
plt.tight_layout()
plt.show()

# Cell 5: Feature Analysis
X, y, _ = drifts['sudden']

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Feature distributions
axes[0].boxplot([X[:, i] for i in range(X.shape[1])])
axes[0].set_xlabel('Feature Index')
axes[0].set_ylabel('Value')
axes[0].set_title('Feature Distributions')
axes[0].grid(True, alpha=0.3)

# Feature-target correlation
correlations = [np.corrcoef(X[:, i], y)[0, 1] for i in range(X.shape[1])]
axes[1].bar(range(X.shape[1]), correlations)
axes[1].set_xlabel('Feature Index')
axes[1].set_ylabel('Correlation with Y')
axes[1].set_title('Feature-Target Correlations')
axes[1].grid(True, axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

# Cell 6: Data Summary Statistics
summary_stats = {
    'Drift Type': [],
    'N Samples': [],
    'Y Mean': [],
    'Y Std': [],
    'Y Min': [],
    'Y Max': []
}

for drift_type, (X, y, _) in drifts.items():
    summary_stats['Drift Type'].append(drift_type)
    summary_stats['N Samples'].append(len(y))
    summary_stats['Y Mean'].append(f"{np.mean(y):.3f}")
    summary_stats['Y Std'].append(f"{np.std(y):.3f}")
    summary_stats['Y Min'].append(f"{np.min(y):.3f}")
    summary_stats['Y Max'].append(f"{np.max(y):.3f}")

summary_df = pd.DataFrame(summary_stats)
print("\nData Summary:")
print(summary_df.to_string(index=False))

# Cell 7: Stationarity Analysis
from scipy import stats as sp_stats

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes = axes.flatten()

for idx, (drift_type, (X, y, true_drifts)) in enumerate(drifts.items()):
    ax = axes[idx]
    
    # Rolling mean and std
    window = 100
    rolling_mean = pd.Series(y).rolling(window=window).mean()
    rolling_std = pd.Series(y).rolling(window=window).std()
    
    ax.plot(y, 'b-', alpha=0.3, linewidth=0.5, label='Original')
    ax.plot(rolling_mean, 'r-', linewidth=2, label=f'Rolling Mean (w={window})')
    ax.fill_between(range(len(y)), 
                    rolling_mean - rolling_std,
                    rolling_mean + rolling_std,
                    alpha=0.2, color='r')
    
    for drift_t in true_drifts:
        ax.axvline(drift_t, color='g', linestyle='--', alpha=0.5, linewidth=2)
    
    ax.set_title(f'{drift_type.capitalize()}: Stationarity Check')
    ax.set_ylabel('Y')
    ax.grid(True, alpha=0.3)
    ax.legend()

axes[-1].set_xlabel('Time')
plt.tight_layout()
plt.show()

print("\n✓ EDA Complete!")